# MedSum-AI — Notebook 06
## Cross-Dataset Comparison & Full Accuracy Report

**Datasets compared**
* Primary  : IU-CXR (English, 3,955 reports)
* Secondary: CASIA-CXR (French, 13,672 reports / 11,111 cleaned)

This notebook **re-runs all models on both datasets** so that the comparison
numbers below are fresh (not loaded from JSON), then generates side-by-side
charts, confusion matrices, a one-page summary dashboard, and a textual
observations report.

The same logic is wrapped as a standalone script in
`src/generate_comparison_report.py`, which writes outputs to
`outputs/comparison_report/`.


## 1. Setup

In [15]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

# Force reimport by removing from sys.modules
if 'generate_comparison_report' in sys.modules:
    del sys.modules['generate_comparison_report']

from generate_comparison_report import (
    load_data, chart_size_coverage, chart_missing, chart_report_length,
    chart_demographics, chart_vocabulary, chart_features,
    run_models_iu, run_models_casia,
    chart_model_comparison, chart_confusion, chart_summary_dashboard,
    write_observations,
)
print('Helpers loaded from src/generate_comparison_report.py')

Helpers loaded from src/generate_comparison_report.py


## 2. Load both datasets

In [16]:
iu_raw, iu_clean, iu_feat, ca_raw, ca_clean, ca_feat = load_data()


[1] Loading both datasets...
  IU-CXR    raw=3,955  clean=2,985  features=2,985
  CASIA-CXR raw=13,672  clean=11,111  features=11,111


## 3. EDA Comparison Charts

Each helper writes the PNG to `outputs/comparison_report/` and returns a
dict of summary statistics.

In [3]:
stats = chart_size_coverage(iu_raw, iu_clean, ca_raw, ca_clean)
stats


[2] Chart: dataset size & coverage
  -> 01_dataset_size_coverage.png


{'iu_raw': 3955,
 'iu_clean': 2985,
 'casia_raw': 13672,
 'casia_clean': 11111,
 'iu_keep_pct': 75.47408343868521,
 'casia_keep_pct': 81.26828554710357}

In [4]:
miss = chart_missing(iu_raw, ca_raw)
print('Top missing columns (IU-CXR):', list(miss['iu_missing_top'].keys())[-3:])
print('Top missing columns (CASIA):', list(miss['casia_missing_top'].keys())[-3:])


[3] Chart: missing-data comparison
  -> 02_missing_data_comparison.png
Top missing columns (IU-CXR): ['findings', 'comparison', 'mesh_minor']
Top missing columns (CASIA): ['Findings', 'Impression', 'Comparison']


In [5]:
len_stats = chart_report_length(iu_clean, ca_clean)
stats.update(len_stats)
len_stats


[4] Chart: report-length distributions
  -> 03_report_length_comparison.png


{'iu_findings_mean': 34.65448892343568,
 'iu_findings_median': 32.0,
 'iu_impression_mean': 12.360187982544478,
 'iu_impression_median': 7.0,
 'casia_findings_mean': 36.87921879218792,
 'casia_findings_median': 35.0,
 'casia_impression_mean': 12.373413734137342,
 'casia_impression_median': 11.0}

In [6]:
demo  = chart_demographics(ca_clean)
vocab = chart_vocabulary(iu_clean, ca_clean)
chart_features(iu_feat, ca_feat)
print('Demographics + vocabulary + feature-overlap charts saved.')


[5] Chart: demographic coverage (CASIA only — IU has no demographics)
  -> 04_demographics_comparison.png

[6] Chart: top-vocabulary comparison
  -> 05_vocabulary_comparison.png

[7] Chart: feature-distribution overlap
  -> 06_feature_distribution_overlap.png
Demographics + vocabulary + feature-overlap charts saved.


## 4. Re-run models (fresh accuracy numbers)

* **IU-CXR** — binary Normal vs Abnormal (Logistic Regression, Random Forest, XGBoost) with 5-fold CV.
* **CASIA-CXR** — 5-class condition classifier + TF-IDF text baseline with 5-fold CV.

In [17]:
iu_results    = run_models_iu(iu_feat)
casia_results = run_models_casia(ca_feat)
print('\nIU-CXR  binary models:'); print(iu_results)
print('\nCASIA-CXR 5-class models:'); print(casia_results)


[8a] Re-training IU-CXR models (Normal vs Abnormal binary)...
  LR  — AUC 0.9600  F1 0.9252
  RF  — AUC 0.9703  F1 0.9559
  XGB — AUC 0.9741  F1 0.9619

[8b] Re-training CASIA-CXR models (5-class condition)...
  LR  — Macro-F1 0.8068
  RF  — Macro-F1 0.9995
  XGB — Macro-F1 0.9994
  LR-TFIDF — Macro-F1 0.9988

IU-CXR  binary models:
{'LR': {'AUC': 0.9600125893165142, 'F1': 0.9251679476730172}, 'RF': {'AUC': 0.9702785794947614, 'F1': 0.9559488519601238}, 'XGB': {'AUC': 0.9740691414477224, 'F1': 0.9619158507394576}, 'cm': [[425, 124], [92, 2344]], 'cm_labels': ['Normal', 'Abnormal']}

CASIA-CXR 5-class models:
{'classes': ['Cardiomegaly', 'Mass', 'PleuralEffusion', 'Pneumonia', 'Pneumothorax'], 'LR': {'Macro_F1': 0.8068163738777805}, 'RF': {'Macro_F1': 0.9994931145181359}, 'XGB': {'Macro_F1': 0.9994162577974801}, 'LR_TFIDF': {'Macro_F1': 0.9987845949006464}, 'cm': [[3756, 0, 0, 0, 0], [0, 1355, 0, 0, 0], [4, 0, 1994, 2, 0], [0, 0, 0, 2000, 0], [0, 0, 0, 0, 2000]], 'cm_labels': ['Cardiom

## 5. Model-accuracy comparison & confusion matrices

In [18]:
chart_model_comparison(iu_results, casia_results)
chart_confusion(iu_results, casia_results)


[9] Chart: model-accuracy comparison
  -> 07_model_accuracy_comparison.png

[10] Chart: confusion-matrix side-by-side
  -> 08_confusion_matrix_comparison.png


## 6. One-page summary dashboard

In [19]:
chart_summary_dashboard(stats, iu_results, casia_results)


[11] Chart: summary dashboard
  -> 09_summary_dashboard.png


## 7. Written observations (auto-generated narrative)

In [20]:
write_observations(stats, vocab, demo, iu_results, casia_results)
print('\n--- observations.txt ---\n')
print(open('../outputs/comparison_report/observations.txt', encoding='utf-8').read())


[12] Writing observations & narrative report
  -> f:\Study\AIML Masters\Capstone Project\MedSUMAI\outputs\comparison_report\REPORT.md
  -> f:\Study\AIML Masters\Capstone Project\MedSUMAI\outputs\comparison_report\observations.txt
  -> comparison_summary.json

--- observations.txt ---

MedSum-AI — Cross-Dataset Observations (auto-generated)

Dataset size
------------
  IU-CXR    raw 3,955  clean 2,985  (75.5% kept)
  CASIA-CXR raw 13,672  clean 11,111  (81.3% kept)
  -> CASIA-CXR adds 3.7x more cleaned records.

Report length (cleaned)
-----------------------
  Findings   IU mean=34.7   CASIA mean=36.9
  Impression IU mean=12.4   CASIA mean=12.4

Demographics
------------
  IU-CXR    : none
  CASIA-CXR : age mean=65.2, median=65.0;
              gender {'M': 6449, 'F': 4639}

Vocabulary
----------
  IU-CXR    : 1,535 unique tokens in Findings
  CASIA-CXR : 480 unique tokens in Findings (smaller, more templated)

Model accuracy (5-fold CV, fresh)
---------------------------------
  IU-C

### Files produced in `outputs/comparison_report/`

* `01_dataset_size_coverage.png`
* `02_missing_data_comparison.png`
* `03_report_length_comparison.png`
* `04_demographics_comparison.png`
* `05_vocabulary_comparison.png`
* `06_feature_distribution_overlap.png`
* `07_model_accuracy_comparison.png`
* `08_confusion_matrix_comparison.png`
* `09_summary_dashboard.png`
* `REPORT.md` — full narrative report with charts inline
* `observations.txt` — plain-text observations
* `comparison_summary.json` — machine-readable summary

The same outputs are produced non-interactively by running:

```bash
python src/generate_comparison_report.py
```